# Week 04 과제 · 모범답안 (교수자용)

> 교수자용 모범답안입니다. 학생에게 배포하지 마세요.

## 준비: 자동 확인 도우미

In [ ]:
import copy

def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")

---
## 문제 1. 단어 빈도 세기 (2점)

`word_count(text)`는 문장에서 각 단어가 몇 번 나오는지 **딕셔너리**로 돌려줍니다.

- 영어는 소문자로 통일합니다.
- 단어 앞뒤의 문장부호 `.,!?"'`는 제거합니다. (`"python!"` → `"python"`)
- 문장부호를 지우고 빈 문자열이 되면 세지 않습니다.

```python
word_count("Python is fun. python IS easy!")
# {"python": 2, "is": 2, "fun": 1, "easy": 1}
```

In [ ]:
def word_count(text):
    """단어별 등장 횟수 딕셔너리를 돌려줍니다."""
    counts = {}
    for raw in text.lower().split():
        word = raw.strip(".,!?\"'")
        if word:
            counts[word] = counts.get(word, 0) + 1
    return counts

In [ ]:
check("영어 문장", lambda: word_count("Python is fun. python IS easy!") == {"python": 2, "is": 2, "fun": 1, "easy": 1})
check("한글 문장", lambda: word_count("목포 바다 목포 항구, 바다!") == {"목포": 2, "바다": 2, "항구": 1})
check("빈 문자열", lambda: word_count("") == {})
check("문장부호만 있는 토큰", lambda: word_count("hi ! hi ...") == {"hi": 2})
check("따옴표 제거", lambda: word_count("'AI' \"ai\" Ai") == {"ai": 3})

---
## 문제 2. 두 수강반 비교 (2점)

`compare_classes(a, b)`는 두 수강반 명단(리스트, 중복 기록이 있을 수 있음)을 받아 다음 딕셔너리를 돌려줍니다.

| 키 | 값 |
|---|---|
| `"both"` | 두 반 모두 듣는 학생 (가나다순 리스트) |
| `"only_a"` | a반만 듣는 학생 (가나다순 리스트) |
| `"only_b"` | b반만 듣는 학생 (가나다순 리스트) |
| `"total"` | 한 반이라도 듣는 학생 수 (정수) |

In [ ]:
def compare_classes(a, b):
    """두 명단의 공통·차이를 딕셔너리로 돌려줍니다."""
    sa, sb = set(a), set(b)
    return {
        "both": sorted(sa & sb),
        "only_a": sorted(sa - sb),
        "only_b": sorted(sb - sa),
        "total": len(sa | sb),
    }

In [ ]:
A = ["김민준", "이서연", "박지호", "김민준", "최하은"]
B = ["최하은", "정도윤", "이서연", "강지우", "정도윤"]
check("both", lambda: compare_classes(A, B)["both"] == ["이서연", "최하은"])
check("only_a", lambda: compare_classes(A, B)["only_a"] == ["김민준", "박지호"])
check("only_b", lambda: compare_classes(A, B)["only_b"] == ["강지우", "정도윤"])
check("total (중복 제외)", lambda: compare_classes(A, B)["total"] == 6)
check("한쪽이 빈 명단", lambda: compare_classes([], ["가"]) == {"both": [], "only_a": [], "only_b": ["가"], "total": 1})

---
## 문제 3. 학과별 성적 요약 (3점)

`dept_summary(records)`는 `{"name", "dept", "score"}` 딕셔너리의 리스트를 받아, 학과마다 다음 정보를 담은 딕셔너리를 돌려줍니다.

```python
{
    "컴퓨터공학과": {"count": 3, "avg": 84.3, "top": "강지우"},
    ...
}
```

- `avg`: 평균을 **소수 첫째 자리**까지 반올림 (`round(값, 1)`)
- `top`: 최고 점수 학생의 이름. **동점이면 이름 가나다순으로 앞선 학생**

In [ ]:
def dept_summary(records):
    """학과별 {count, avg, top} 딕셔너리를 돌려줍니다."""
    groups = {}
    for r in records:
        groups.setdefault(r["dept"], []).append(r)
    result = {}
    for dept, members in groups.items():
        scores = [m["score"] for m in members]
        best = sorted(members, key=lambda m: (-m["score"], m["name"]))[0]
        result[dept] = {"count": len(members), "avg": round(sum(scores) / len(scores), 1), "top": best["name"]}
    return result

In [ ]:
recs = [
    {"name": "김민준", "dept": "컴퓨터공학과", "score": 80},
    {"name": "이서연", "dept": "정보보안학과", "score": 91},
    {"name": "강지우", "dept": "컴퓨터공학과", "score": 95},
    {"name": "박지호", "dept": "컴퓨터공학과", "score": 78},
    {"name": "정도윤", "dept": "정보보안학과", "score": 91},
    {"name": "최하은", "dept": "경영학과", "score": 66},
]
check("학과 수", lambda: set(dept_summary(recs)) == {"컴퓨터공학과", "정보보안학과", "경영학과"})
check("컴퓨터공학과 요약", lambda: dept_summary(recs)["컴퓨터공학과"] == {"count": 3, "avg": 84.3, "top": "강지우"})
check("동점이면 가나다순 (이서연 vs 정도윤)", lambda: dept_summary(recs)["정보보안학과"]["top"] == "이서연")
check("한 명뿐인 학과", lambda: dept_summary(recs)["경영학과"] == {"count": 1, "avg": 66.0, "top": "최하은"})
check("빈 목록", lambda: dept_summary([]) == {})

---
## 문제 4. 순위와 필터 – 원본을 지켜라 (AI 협업, 3점)

두 함수를 완성하세요. **두 함수 모두 입력 리스트(와 그 안의 딕셔너리)를 절대 바꾸면 안 됩니다.**

1. `top_n(records, n)`: 점수 내림차순, 동점이면 이름 가나다순으로 정렬해 **상위 n명의 이름 리스트**를 돌려줍니다. n이 인원보다 크면 전원을 돌려줍니다.
2. `drop_below(records, cutoff)`: 점수가 `cutoff` **미만**인 레코드를 뺀 **새 리스트**를 돌려줍니다. 순서는 원래 순서를 유지합니다.

**AI 협업 과제**: AI에게 먼저 "점수 낮은 학생을 리스트에서 지우는 함수"를 요청해 보고, 받은 코드가 (1) 연속된 삭제 대상에서 건너뛰지 않는지, (2) 원본을 바꾸지 않는지 자동 확인으로 검증하세요.

In [ ]:
def top_n(records, n):
    """상위 n명의 이름 리스트 (점수 내림차순, 동점이면 이름순)."""
    ordered = sorted(records, key=lambda r: (-r["score"], r["name"]))
    return [r["name"] for r in ordered[:n]]


def drop_below(records, cutoff):
    """cutoff 미만을 뺀 새 리스트 (원래 순서 유지, 원본 불변)."""
    return [r for r in records if r["score"] >= cutoff]

In [ ]:
before = copy.deepcopy(recs)
check("top_n 상위 3명", lambda: top_n(recs, 3) == ["강지우", "이서연", "정도윤"])
check("top_n n이 인원보다 큼", lambda: len(top_n(recs, 100)) == len(recs))
check("top_n 후 원본 불변", lambda: (top_n(recs, 3), recs == before)[1])

low_first = [{"name": "가", "dept": "X", "score": 50}, {"name": "나", "dept": "X", "score": 55},
             {"name": "다", "dept": "X", "score": 90}, {"name": "라", "dept": "X", "score": 40},
             {"name": "마", "dept": "X", "score": 70}]
low_before = copy.deepcopy(low_first)
check("drop_below 연속 삭제 대상", lambda: [r["name"] for r in drop_below(low_first, 60)] == ["다", "마"])
check("drop_below 후 원본 불변", lambda: (drop_below(low_first, 60), low_first == low_before)[1])
check("경계값: cutoff와 같은 점수는 남김", lambda: [r["name"] for r in drop_below(low_first, 55)] == ["나", "다", "마"])
check("반환값이 원본과 다른 리스트 객체", lambda: drop_below(low_first, 0) is not low_first)

### AI 활용 기록 (예시)

| 항목 | 내용 |
|---|---|
| 사용한 AI 도구 | (예) Claude |
| 처음 프롬프트와 문제 | "점수 낮은 학생을 지우는 함수" → `for r in records: if ...: records.remove(r)` 코드. 연속 삭제 대상에서 '나'가 남았고 원본도 바뀌었음 |
| 고친 프롬프트 | "원본 리스트는 바꾸지 말고, cutoff 미만을 제외한 새 리스트를 원래 순서대로 반환. 동점은 이름 가나다순" |
| 직접 고친 부분 | 리스트 컴프리헨션으로 새 리스트 생성, 정렬 key를 `(-score, name)` 튜플로 |

---
## 성찰 (필수)

1. 네 가지 자료구조 중 이번 과제에서 가장 유용했던 것과 이유는?
2. "원본을 바꾸지 않는다"는 조건이 왜 중요할까요? 실제로 원본이 바뀐 경험이 있었나요?
3. AI 코드의 함정 중 직접 겪은 것은 무엇인가요?

(학생 작성 영역)